In [ ]:
"""
DINOv2 + motion signature for Classifier B on full DoTA.

Inputs (attach all three):
  /kaggle/input/datasets/antonygerold6/dota-tracks-v4/tracks
  /kaggle/input/datasets/uom200399g/dota-dataset
  /kaggle/input/datasets/antonygerold6/dota-session2

Settings: GPU T4 x1, Save Version > Advanced > Always save output ON.
Total runtime: ~90 min. Embeddings cached to /kaggle/working/dino_embs.npy.
"""
import os, json, sys, time, warnings
from pathlib import Path
import subprocess

warnings.filterwarnings('ignore')

print("[INSTALL]", flush=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightgbm", "pyarrow", "timm", "opencv-python-headless"],
                   capture_output=True, text=True)
if r.returncode != 0: print(r.stderr); sys.exit(1)
print("ok", flush=True)

import numpy as np
import pandas as pd
import torch
import cv2
import timm
from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import lightgbm as lgb

TRACKS_DIR = Path("/kaggle/input/datasets/antonygerold6/dota-tracks-v4/tracks")
ANN_DIR = Path("/kaggle/input/datasets/uom200399g/dota-dataset/dota-anno/annotations")
FRAMES_DIR = Path("/kaggle/input/datasets/uom200399g/dota-dataset/dota/frames")
FEATURES_PATH = Path("/kaggle/input/datasets/antonygerold6/dota-session2/features_v3_full.parquet")
PREDS_PATH = Path("/kaggle/input/datasets/antonygerold6/dota-session2/classifier_a_v4_clean_predictions_full.parquet")
OUT_DIR = Path("/kaggle/working")

assert TRACKS_DIR.exists() and ANN_DIR.exists() and FRAMES_DIR.exists()
assert FEATURES_PATH.exists() and PREDS_PATH.exists()
assert torch.cuda.is_available()
device = torch.device('cuda')
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

# ============================================================
t0 = time.time()
print("\n[1/9] Load features", flush=True)
df = pd.read_parquet(FEATURES_PATH)
pred_v4 = pd.read_parquet(PREDS_PATH)
print(f"      df={df.shape}  pred={pred_v4.shape}  ({time.time()-t0:.1f}s)", flush=True)

# ============================================================
t = time.time()
print("\n[2/9] Build lookups", flush=True)
keys = list(zip(pred_v4['video'].values, pred_v4['frame'].values, pred_v4['track_id'].values))
clean_lookup = dict(zip(keys, pred_v4['true_label_clean'].values))
prob_lookup = dict(zip(keys, pred_v4['prob_anomaly'].values))
del pred_v4, keys
df_keys = list(zip(df['video'].values, df['frame'].values, df['track_id'].values))
df['is_anomalous'] = [clean_lookup.get(k, 0) for k in df_keys]
del clean_lookup, df_keys
print(f"      anomalous={df['is_anomalous'].sum():,}  ({time.time()-t:.1f}s)", flush=True)

# ============================================================
t = time.time()
print("\n[3/9] Load annotations", flush=True)
annotations = {}
for jf in ANN_DIR.glob("*.json"):
    a = json.load(open(jf))
    annotations[a['video_name']] = a
print(f"      {len(annotations)}  ({time.time()-t:.1f}s)", flush=True)

# ============================================================
t = time.time()
print("\n[4/9] Anomalous track list", flush=True)
df = df.sort_values(['video','track_id','frame']).reset_index(drop=True)
KEEP = {'turning','lateral','moving_ahead_or_waiting','oncoming',
        'leave_to_right','leave_to_left','pedestrian','start_stop_or_stationary'}

track_rows = []
for vid, vdf in df.groupby('video', sort=False):
    if vid not in annotations: continue
    ann = annotations[vid]
    cls = ann.get('accident_name', 'unknown')
    if cls not in KEEP: continue
    anom_s, anom_e = ann['anomaly_start'], ann['anomaly_end']
    for tid, tdf in vdf.groupby('track_id'):
        win = tdf[(tdf['frame'] >= anom_s) & (tdf['frame'] <= anom_e)]
        if len(win) == 0: continue
        if win['is_anomalous'].max() != 1: continue
        wframes = sorted(win['frame'].unique())
        if len(wframes) < 5:
            sample_frames = wframes
        else:
            idx = np.linspace(0, len(wframes)-1, 5).astype(int)
            sample_frames = [int(wframes[i]) for i in idx]
        track_rows.append({'video': vid, 'track_id': int(tid),
                           'class': cls, 'anom_start': anom_s, 'anom_end': anom_e,
                           'sample_frames': sample_frames})
tldf = pd.DataFrame(track_rows)
print(f"      tracks={len(tldf)}  ({time.time()-t:.1f}s)", flush=True)
print(f"      class dist:", flush=True)
for cls, n in tldf['class'].value_counts().items():
    print(f"        {cls:<30} {n}", flush=True)

# ============================================================
t = time.time()
print("\n[5/9] Build bbox lookup from track JSONs", flush=True)
bbox_lookup = {}
for tp in TRACKS_DIR.glob("*.json"):
    vid = tp.stem
    for f in json.load(open(tp)):
        for d in f.get('detections', []):
            bbox_lookup[(vid, f['frame'], d['track_id'])] = d['bbox']
print(f"      bbox lookup={len(bbox_lookup):,}  ({time.time()-t:.1f}s)", flush=True)

# ============================================================
print("\n[6/9] DINOv2 embeddings", flush=True)
EMBS_PATH = OUT_DIR / "dino_embs.npy"
INDEX_PATH = OUT_DIR / "dino_index.json"

def run_embedding():
    print("      loading DINOv2-small...", flush=True)
    model = timm.create_model('vit_small_patch14_dinov2.lvd142m',
                              pretrained=True, num_classes=0)
    model = model.eval().to(device).half()
    EMBED_DIM = 384
    INPUT_SIZE = 518   # FIXED: DINOv2 expects 518, not 224
    BATCH = 32         # FIXED: smaller batch for larger input
    mean = torch.tensor([0.485, 0.456, 0.406], device=device, dtype=torch.float16).view(1,3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device, dtype=torch.float16).view(1,3,1,1)

    targets = []
    for tidx, row in tldf.iterrows():
        for fid in row['sample_frames']:
            k = (row['video'], int(fid), int(row['track_id']))
            if k in bbox_lookup:
                targets.append((tidx, row['video'], int(fid), int(row['track_id'])))
    print(f"      total crops: {len(targets):,}  ({len(targets)*2:,} forward passes)", flush=True)
    embs_obj = np.zeros((len(targets), EMBED_DIM), dtype=np.float32)
    embs_ctx = np.zeros((len(targets), EMBED_DIM), dtype=np.float32)

    by_frame = defaultdict(list)
    for i, (tidx, vid, fid, tid) in enumerate(targets):
        by_frame[(vid, fid)].append((i, tid))
    frame_keys = list(by_frame.keys())
    print(f"      unique frames: {len(frame_keys):,}", flush=True)

    batch_obj_crops = []
    batch_ctx_crops = []
    batch_indices = []

    def flush():
        nonlocal batch_obj_crops, batch_ctx_crops, batch_indices
        if not batch_obj_crops: return
        obj_stack = torch.stack(batch_obj_crops).to(device).half()
        ctx_stack = torch.stack(batch_ctx_crops).to(device).half()
        obj_stack = (obj_stack - mean) / std
        ctx_stack = (ctx_stack - mean) / std
        with torch.no_grad():
            f_obj = model(obj_stack).float().cpu().numpy()
            f_ctx = model(ctx_stack).float().cpu().numpy()
        for j, ii in enumerate(batch_indices):
            embs_obj[ii] = f_obj[j]
            embs_ctx[ii] = f_ctx[j]
        batch_obj_crops = []
        batch_ctx_crops = []
        batch_indices = []

    t_emb = time.time()
    n_frames_loaded = 0
    n_frames_failed = 0
    for fkidx, (vid, fid) in enumerate(frame_keys, 1):
        items = by_frame[(vid, fid)]
        img_path = FRAMES_DIR / vid / "images" / f"{fid:06d}.jpg"
        if not img_path.exists():
            cand = list((FRAMES_DIR / vid).rglob(f"*{fid:06d}*.jpg"))
            if cand: img_path = cand[0]
            else:
                n_frames_failed += 1
                continue
        img = cv2.imread(str(img_path))
        if img is None:
            n_frames_failed += 1
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        n_frames_loaded += 1

        for ii, tid in items:
            bbox = bbox_lookup.get((vid, fid, tid))
            if bbox is None: continue
            x1, y1, x2, y2 = [int(v) for v in bbox]
            x1c = max(0, x1); y1c = max(0, y1)
            x2c = min(W, x2); y2c = min(H, y2)
            if x2c <= x1c or y2c <= y1c: continue
            obj = img[y1c:y2c, x1c:x2c]
            if obj.size == 0: continue
            h, w = obj.shape[:2]
            s = max(h, w)
            pad = np.full((s, s, 3), int(obj.mean()), dtype=np.uint8)
            pad[(s-h)//2:(s-h)//2+h, (s-w)//2:(s-w)//2+w] = obj
            obj_resized = cv2.resize(pad, (INPUT_SIZE, INPUT_SIZE))

            bw = x2 - x1; bh = y2 - y1
            cx_b = (x1 + x2) // 2; cy_b = (y1 + y2) // 2
            cw = int(bw * 1.5); ch = int(bh * 1.5)
            cx1 = max(0, cx_b - cw//2); cy1 = max(0, cy_b - ch//2)
            cx2 = min(W, cx_b + cw//2); cy2 = min(H, cy_b + ch//2)
            ctx = img[cy1:cy2, cx1:cx2]
            if ctx.size == 0: continue
            h2, w2 = ctx.shape[:2]
            s2 = max(h2, w2)
            pad2 = np.full((s2, s2, 3), int(ctx.mean()), dtype=np.uint8)
            pad2[(s2-h2)//2:(s2-h2)//2+h2, (s2-w2)//2:(s2-w2)//2+w2] = ctx
            ctx_resized = cv2.resize(pad2, (INPUT_SIZE, INPUT_SIZE))

            obj_t = torch.from_numpy(obj_resized).permute(2,0,1).float() / 255.0
            ctx_t = torch.from_numpy(ctx_resized).permute(2,0,1).float() / 255.0
            batch_obj_crops.append(obj_t)
            batch_ctx_crops.append(ctx_t)
            batch_indices.append(ii)
            if len(batch_obj_crops) >= BATCH:
                flush()

        if fkidx % 5000 == 0:
            elapsed = (time.time() - t_emb) / 60
            rate = fkidx / max(1, time.time()-t_emb)
            eta = (len(frame_keys) - fkidx) / max(1e-6, rate) / 60
            print(f"        frames [{fkidx}/{len(frame_keys)}]  {elapsed:.1f}m  eta {eta:.1f}m  failed={n_frames_failed}", flush=True)
    flush()
    print(f"      embedding done ({(time.time()-t_emb)/60:.1f}m)  loaded={n_frames_loaded} failed={n_frames_failed}", flush=True)

    track_embs = np.zeros((len(tldf), EMBED_DIM*2), dtype=np.float32)
    track_counts = np.zeros(len(tldf), dtype=int)
    for i, (tidx, vid, fid, tid) in enumerate(targets):
        track_embs[tidx, :EMBED_DIM] += embs_obj[i]
        track_embs[tidx, EMBED_DIM:] += embs_ctx[i]
        track_counts[tidx] += 1
    valid_local = track_counts > 0
    track_embs[valid_local] /= track_counts[valid_local, None]
    print(f"      tracks with embeddings: {valid_local.sum()}/{len(tldf)}", flush=True)

    np.save(EMBS_PATH, track_embs)
    json.dump({'valid': valid_local.tolist(), 'n_tracks': int(len(tldf))}, open(INDEX_PATH, 'w'))
    print(f"      saved cache to {EMBS_PATH}", flush=True)
    return track_embs

if EMBS_PATH.exists() and INDEX_PATH.exists():
    print("      cached found, loading...", flush=True)
    embs = np.load(EMBS_PATH)
    print(f"      embs={embs.shape}", flush=True)
else:
    embs = run_embedding()

valid_emb = np.any(embs != 0, axis=1)
print(f"      tracks with valid embeddings: {valid_emb.sum()}/{len(tldf)}", flush=True)

# ============================================================
t = time.time()
print("\n[7/9] PCA on DINOv2 embeddings", flush=True)
PCA_DIM = 128
pca = PCA(n_components=PCA_DIM, random_state=42)
pca.fit(embs[valid_emb])
embs_pca = pca.transform(embs)
print(f"      PCA {embs.shape[1]}->{PCA_DIM}, variance retained: {pca.explained_variance_ratio_.sum():.3f}  ({time.time()-t:.1f}s)", flush=True)

# ============================================================
print("\n[8/9] Motion signatures", flush=True)
t = time.time()

def fourier_descriptor(traj, n_coeffs=4):
    if len(traj) < 4: return np.zeros(2*n_coeffs, dtype=np.float32)
    old_idx = np.linspace(0, 1, len(traj)); new_idx = np.linspace(0, 1, 32)
    rx = np.interp(new_idx, old_idx, traj[:,0]); ry = np.interp(new_idx, old_idx, traj[:,1])
    z = (rx-rx.mean()) + 1j*(ry-ry.mean())
    F = np.fft.fft(z); norm = abs(F[1]) + 1e-9
    return np.concatenate([np.abs(F[1:n_coeffs+1])/norm, np.angle(F[1:n_coeffs+1])]).astype(np.float32)

def direction_histogram(traj, n_bins=8):
    if len(traj) < 2: return np.zeros(n_bins, dtype=np.float32)
    d = np.diff(traj, axis=0)
    angles = np.arctan2(d[:,1], d[:,0])
    hist, _ = np.histogram(angles, bins=n_bins, range=(-np.pi, np.pi))
    return (hist / max(1, hist.sum())).astype(np.float32)

def resample_curve(values, n_bins=10):
    if len(values) < 2: return np.full(n_bins, float(np.mean(values) if len(values) else 0), dtype=np.float32)
    return np.interp(np.linspace(0,1,n_bins), np.linspace(0,1,len(values)), values).astype(np.float32)

print("      pre-compute frame context...", flush=True)
context_per_frame = {}
for (vid, fid), fdf_g in df.groupby(['video','frame'], sort=False):
    if len(fdf_g) < 2: continue
    angles = np.arctan2(fdf_g['vy_norm'].values, fdf_g['vx_norm'].values)
    hist, _ = np.histogram(angles, bins=8, range=(-np.pi, np.pi))
    context_per_frame[(vid, int(fid))] = hist.astype(float) / max(1, hist.sum())

print(f"      indexing df...", flush=True)
df_indexed = df.set_index(['video','track_id']).sort_index()

motion_sigs = []
valid_motion = []
for idx, row in tldf.iterrows():
    vid, tid = row['video'], row['track_id']
    anom_s, anom_e = row['anom_start'], row['anom_end']
    try:
        tdf = df_indexed.loc[(vid, tid)]
        if isinstance(tdf, pd.Series): tdf = tdf.to_frame().T
        tdf = tdf.sort_values('frame')
    except KeyError:
        valid_motion.append(False); motion_sigs.append(np.zeros(64, dtype=np.float32)); continue
    win = tdf[(tdf['frame'] >= anom_s) & (tdf['frame'] <= anom_e)]
    if len(win) < 2:
        valid_motion.append(False); motion_sigs.append(np.zeros(64, dtype=np.float32)); continue

    traj = win[['cx','cy']].values
    fd = fourier_descriptor(traj, 4); dh = direction_histogram(traj, 8)
    sp = win['speed_norm'].values
    speed_stats = [float(sp[0]), float(sp[-1]), float(sp.max()), float(sp.mean())]
    cx_s, cy_s = float(win['cx'].iloc[0]), float(win['cy'].iloc[0])
    cx_e, cy_e = float(win['cx'].iloc[-1]), float(win['cy'].iloc[-1])
    endpoint = [cx_s, cy_s, cx_e, cy_e, cx_e-cx_s, cy_e-cy_s]
    duration = [len(win), len(win)/max(1, len(tdf))]
    v4_scores = [prob_lookup.get((vid, int(fr), int(tid)), 0.0) for fr in win['frame'].values]
    v4_curve = resample_curve(v4_scores, 10)
    v4_stats = [float(np.mean(v4_scores)), float(np.max(v4_scores)),
                float(np.argmax(v4_scores))/max(1,len(v4_scores)-1) if v4_scores else 0.0]
    other_dirs = []
    for fr in win['frame'].values:
        c = context_per_frame.get((vid, int(fr)))
        if c is not None: other_dirs.append(c)
    if other_dirs:
        context_dir = np.mean(other_dirs, axis=0).astype(np.float32)
        alignment = float(np.dot(dh, context_dir))
    else:
        context_dir = np.zeros(8, dtype=np.float32); alignment = 0.5
    if 'inverse_ttc' in win.columns and len(win)>1:
        itc = win['inverse_ttc'].values
        peak_idx = int(np.argmax(itc))
        ttc_feats = [peak_idx/max(1,len(win)-1), float(itc[peak_idx])]
    else:
        ttc_feats = [0.5, 0.0]
    interact = [float(win['nn_closing_max'].max()) if 'nn_closing_max' in win.columns else 0,
                float(win['nn_dist_min'].min()) if 'nn_dist_min' in win.columns else 1,
                float(win['n_others_in_frame'].mean()) if 'n_others_in_frame' in win.columns else 0]
    size = [float(win['area_norm'].iloc[0]), float(win['area_norm'].iloc[-1]),
            float(win['area_norm'].iloc[-1])/max(1e-6, float(win['area_norm'].iloc[0])),
            float(win['aspect_ratio'].iloc[0]), float(win['aspect_ratio'].iloc[-1])]
    if 'jerk_norm' in win.columns:
        j = win['jerk_norm'].values
        jerk = [float(np.mean(j)), float(np.max(j)), float(np.std(j)) if len(j)>1 else 0]
    else: jerk = [0,0,0]
    yolo_cls = int(win['yolo_class'].mode()[0]) if len(win) else -1
    feats = np.concatenate([
        fd, dh, speed_stats, endpoint, duration, v4_curve, v4_stats,
        context_dir, [alignment], ttc_feats, interact, size, jerk, [yolo_cls]
    ]).astype(np.float32)
    motion_sigs.append(feats); valid_motion.append(True)

motion_X = np.stack(motion_sigs)
valid_motion = np.array(valid_motion)
both_valid = valid_motion & valid_emb
print(f"      tracks with both motion+dino valid: {both_valid.sum()}/{len(tldf)}  ({time.time()-t:.1f}s)", flush=True)

tldf_v = tldf[both_valid].reset_index(drop=True)
motion_X = motion_X[both_valid]
embs_pca_v = embs_pca[both_valid]
fused_X = np.concatenate([motion_X, embs_pca_v], axis=1)

# ============================================================
print("\n[9/9] Classifier B with three feature sets + ensemble", flush=True)
classes = sorted(tldf_v['class'].unique())
cls_to_idx = {c: i for i, c in enumerate(classes)}
y = tldf_v['class'].map(cls_to_idx).values
K = len(classes)
turning_idx = cls_to_idx.get('turning')

vid_list = sorted(tldf_v['video'].unique())
vid_to_cls = tldf_v.groupby('video')['class'].first().to_dict()
vid_strat = [cls_to_idx[vid_to_cls[v]] for v in vid_list]
try:
    splits = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(vid_list, vid_strat))
except ValueError:
    splits = list(KFold(n_splits=5, shuffle=True, random_state=42).split(vid_list))

def balanced_indices(y_tr, target=2500, seed=42):
    rng = np.random.default_rng(seed)
    picked = []
    for c in range(K):
        idx = np.where(y_tr == c)[0]
        if len(idx) == 0: continue
        if len(idx) > target:
            picked.extend(rng.choice(idx, target, replace=False).tolist())
        else:
            picked.extend(idx.tolist())
            extra = target - len(idx)
            if extra > 0: picked.extend(rng.choice(idx, extra, replace=True).tolist())
    return np.array(picked)

def two_stage_a3(X_data, label):
    top1s, top3s = [], []
    per_cls = {c: [0,0] for c in classes}
    oof_probs = np.zeros((len(tldf_v), K))
    for fi, (tr_vi, te_vi) in enumerate(splits):
        t_f = time.time()
        tr_vids = {vid_list[i] for i in tr_vi}; te_vids = {vid_list[i] for i in te_vi}
        tr_m = tldf_v['video'].isin(tr_vids).values; te_m = tldf_v['video'].isin(te_vids).values
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_data[tr_m]); X_te = scaler.transform(X_data[te_m])
        y_tr = y[tr_m]; y_te = y[te_m]
        y_s1 = (y_tr == turning_idx).astype(int)
        pw = (y_s1==0).sum() / max(1, (y_s1==1).sum())
        m1 = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                                num_leaves=31, min_child_samples=20,
                                scale_pos_weight=pw, n_jobs=-1, random_state=42, verbose=-1)
        m1.fit(X_tr, y_s1)
        p_turn = m1.predict_proba(X_te)[:, 1]
        not_turn = y_tr != turning_idx
        X_tr2 = X_tr[not_turn]; y_tr2 = y_tr[not_turn]
        bal = balanced_indices(y_tr2, 1500, 42+fi)
        m2 = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, max_depth=6,
                                num_leaves=63, min_child_samples=20,
                                n_jobs=-1, random_state=42, verbose=-1)
        m2.fit(X_tr2[bal], y_tr2[bal])
        p_fine_raw = m2.predict_proba(X_te)
        p_fine = np.zeros((len(X_te), K))
        for ci, ck in enumerate(m2.classes_): p_fine[:, ck] = p_fine_raw[:, ci]
        p = np.zeros((len(X_te), K))
        p[:, turning_idx] = p_turn
        for c_idx in range(K):
            if c_idx == turning_idx: continue
            p[:, c_idx] = (1 - p_turn) * p_fine[:, c_idx]
        p = p / (p.sum(axis=1, keepdims=True) + 1e-9)
        oof_probs[te_m] = p
        preds = np.argmax(p, axis=1)
        top1 = (preds == y_te).mean()
        top3 = np.mean([y_te[i] in np.argsort(p[i])[::-1][:3] for i in range(len(y_te))])
        top1s.append(top1); top3s.append(top3)
        for i, yv in enumerate(y_te):
            per_cls[classes[yv]][1] += 1
            if preds[i] == yv: per_cls[classes[yv]][0] += 1
        print(f"        [{label}] fold {fi+1}: top-1={top1:.3f} top-3={top3:.3f} ({time.time()-t_f:.1f}s)", flush=True)
    return top1s, top3s, per_cls, oof_probs

print("\n      Motion-only", flush=True)
t1_m, t3_m, pc_m, p_m = two_stage_a3(motion_X, 'motion')
print(f"      motion-only: top-1={np.mean(t1_m):.3f}  top-3={np.mean(t3_m):.3f}", flush=True)

print("\n      DINOv2-only", flush=True)
t1_d, t3_d, pc_d, p_d = two_stage_a3(embs_pca_v, 'dino')
print(f"      dino-only: top-1={np.mean(t1_d):.3f}  top-3={np.mean(t3_d):.3f}", flush=True)

print("\n      Fused (motion + DINO concat)", flush=True)
t1_f, t3_f, pc_f, p_f = two_stage_a3(fused_X, 'fused')
print(f"      fused: top-1={np.mean(t1_f):.3f}  top-3={np.mean(t3_f):.3f}", flush=True)

print("\n      Late-fusion ensemble (avg motion + dino probs)", flush=True)
p_ens = (p_m + p_d) / 2
preds_ens = np.argmax(p_ens, axis=1)
ens_top1 = (preds_ens == y).mean()
ens_top3 = np.mean([y[i] in np.argsort(p_ens[i])[::-1][:3] for i in range(len(y))])
pc_ens = {c: [0,0] for c in classes}
for i, yv in enumerate(y):
    pc_ens[classes[yv]][1] += 1
    if preds_ens[i] == yv: pc_ens[classes[yv]][0] += 1
print(f"      ensemble: top-1={ens_top1:.3f}  top-3={ens_top3:.3f}", flush=True)

# ============================================================
print("\n" + "="*72); print("FINAL"); print("="*72)
print(f"{'Method':<35} {'Top-1':>8} {'Top-3':>8}")
print("-"*53)
print(f"{'v3 (motion only, all 10 cls)':<35} {0.439:>8.3f} {0.779:>8.3f}")
print(f"{'A3 motion only (8 cls)':<35} {np.mean(t1_m):>8.3f} {np.mean(t3_m):>8.3f}")
print(f"{'A3 DINO only':<35} {np.mean(t1_d):>8.3f} {np.mean(t3_d):>8.3f}")
print(f"{'A3 fused (motion+DINO)':<35} {np.mean(t1_f):>8.3f} {np.mean(t3_f):>8.3f}")
print(f"{'Late-fusion ensemble':<35} {ens_top1:>8.3f} {ens_top3:>8.3f}")

best_name, best_top1, best_pc = 'motion', np.mean(t1_m), pc_m
for name, t1, pc in [('dino', np.mean(t1_d), pc_d), ('fused', np.mean(t1_f), pc_f), ('ensemble', ens_top1, pc_ens)]:
    if t1 > best_top1: best_name, best_top1, best_pc = name, t1, pc

print(f"\nBest: {best_name}")
print(f"\nPer-class accuracy ({best_name}):")
print(f"  {'Class':<28} {'Support':>8} {'Top-1':>8}")
print("-"*50)
for cls in sorted(best_pc, key=lambda c: -best_pc[c][1]):
    c, t = best_pc[cls]
    if t == 0: continue
    print(f"  {cls:<28} {t:>8} {c/t:>8.3f}")

json.dump({
    'n_tracks': len(tldf_v),
    'motion': {'top1': float(np.mean(t1_m)), 'top3': float(np.mean(t3_m))},
    'dino':   {'top1': float(np.mean(t1_d)), 'top3': float(np.mean(t3_d))},
    'fused':  {'top1': float(np.mean(t1_f)), 'top3': float(np.mean(t3_f))},
    'ensemble': {'top1': float(ens_top1), 'top3': float(ens_top3)},
    'best': best_name,
    'best_per_class': {c: {'support': best_pc[c][1], 'correct': best_pc[c][0],
                           'accuracy': best_pc[c][0]/best_pc[c][1] if best_pc[c][1] else 0}
                       for c in classes},
    'pca_variance': float(pca.explained_variance_ratio_.sum()),
}, open(OUT_DIR / "classifier_b_dino_results.json", 'w'), indent=2)
print(f"\nSaved: {OUT_DIR / 'classifier_b_dino_results.json'}")
print(f"Total: {(time.time()-t0)/60:.1f}m")
print("DONE")